In [1]:
import time
import requests
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

### **Dados**

In [2]:
# baixa dataset diretamente do repositório no github
GITHUB_LINK = "https://raw.githubusercontent.com/volneiklehm/highlights-app/88e31f419e2120e55581ee50ed99f33602cd16e3/data/News_Category_Dataset_balanced.csv"
response = requests.get(GITHUB_LINK)

DATASET_FILE = "News_Category_Dataset_balanced.csv"
with open(DATASET_FILE, 'wb') as f:
    f.write(response.content)

In [3]:
df = pd.read_csv(DATASET_FILE)
df

,headline,category
0,kanye west's 'erratic' medication dosage repor...,ENTERTAINMENT
1,regina king reveals her plan to push diversity...,ENTERTAINMENT
2,a 13-year-old girl was largely responsible for...,ENTERTAINMENT
3,youtuber jake paul charged with trespassing in...,ENTERTAINMENT
4,a woman proposed to her girlfriend during ryan...,ENTERTAINMENT
...,...,...
58879,get over your fear of not being good enough! (...,WELLNESS
58880,thanksgiving thanks for a healthy life,WELLNESS
58881,"drug safety tracked even after approval, fda says",WELLNESS
58882,television rehab: treatment or trauma?,WELLNESS


In [4]:
df["category"].value_counts()

category
ENTERTAINMENT     9814
OTHER             9814
POLITICS          9814
STYLE & BEAUTY    9814
TRAVEL            9814
WELLNESS          9814
Name: count, dtype: int64

In [5]:
# remove todas as linhas com category 'OTHER'
df = df[df["category"] != "OTHER"]
df["category"].value_counts()

category
ENTERTAINMENT     9814
POLITICS          9814
STYLE & BEAUTY    9814
TRAVEL            9814
WELLNESS          9814
Name: count, dtype: int64

In [6]:
NUM_LABELS = df["category"].nunique() # Número de categorias únicas
print(f"Número de categorias únicas: {NUM_LABELS}")

Número de categorias únicas: 5


### **Pré-processamento de Texto para TF-IDF**

Antes de aplicar o TF-IDF, é uma boa prática pré-processar o texto para remover ruídos e normalizar as palavras. Isso pode incluir:

1.  **Converter para minúsculas**: Trata 'Palavra' e 'palavra' como a mesma.
2.  **Remover pontuação**: Caracteres como vírgulas, pontos e exclamações geralmente não adicionam valor semântico.
3.  **Remover números**: Dependendo da tarefa, números podem ser irrelevantes.
4.  **Remover stopwords**: Palavras muito comuns (como 'a', 'o', 'de', 'e') que não carregam muito significado. O `TfidfVectorizer` já faz isso se `stop_words='english'` for configurado, mas uma etapa explícita pode ser mais customizável.
5.  **Stemming ou Lemmatização**: Reduzir palavras às suas formas base (ex: 'correndo', 'correu' -> 'correr'). Isso pode agrupar termos relacionados, mas deve ser usado com cautela, pois pode alterar o sentido de algumas palavras.

In [7]:
! pip install nltk

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   --------------------------------- ------ 1.3/1.6 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 6.5 MB/s  0:00:00


In [8]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Baixa os recursos do NLTK se ainda não tiverem sido baixados
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

# Define a lista de stopwords em inglês
stop_words = set(stopwords.words('english'))

# Inicializa o Porter Stemmer
# stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()  # 1. Converter para minúsculas
    # 2. Remover pontuação e caracteres não alfanuméricos
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text) # 3. Remover números

    # 4. Remover stopwords e aplicar stemming
    words = text.split()
    # words = [stemmer.stem(word) for word in words if word not in stop_words] # Comentado o stemming
    words = [word for word in words if word not in stop_words] # Apenas remove stopwords
    text = ' '.join(words)

    return text

# Aplica a função de pré-processamento à coluna 'headline'
df['clean_headline'] = df['headline'].apply(preprocess_text)

print("Headlines originais vs. pré-processados (primeiras 5 amostras):")
for i in range(5):
    print(f"Original: {df['headline'].iloc[i]}")
    print(f"Limpo:    {df['clean_headline'].iloc[i]}\n")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\eduar\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Headlines originais vs. pré-processados (primeiras 5 amostras):
Original: kanye west's 'erratic' medication dosage reportedly led to hospitalization
Limpo:    kanye wests erratic medication dosage reportedly led hospitalization

Original: regina king reveals her plan to push diversity at cannes film festival
Limpo:    regina king reveals plan push diversity cannes film festival

Original: a 13-year-old girl was largely responsible for starting johnny depp's career
Limpo:    yearold girl largely responsible starting johnny depps career

Original: youtuber jake paul charged with trespassing in looted arizona mall
Limpo:    youtuber jake paul charged trespassing looted arizona mall

Original: a woman proposed to her girlfriend during ryan gosling's directorial debut
Limpo:    woman proposed girlfriend ryan goslings directorial debut



C:\Users\eduar\AppData\Local\Temp\ipykernel_13688\3841145334.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['clean_headline'] = df['headline'].apply(preprocess_text)


In [15]:
X = df["headline"]
# X = df["clean_headline"] # usa a coluna pré-processada
y = df["category"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Treino: {len(X_train)} amostras | Teste: {len(X_test)} amostras")

Treino: 39256 amostras | Teste: 9814 amostras


### **Modelo com TD-IDF + LogisticRegression**

In [16]:
# ════════════════════════════════════════════════════════════════════════════
# MUDE ESTES PARÂMETROS A CADA EXPERIMENTO
# ════════════════════════════════════════════════════════════════════════════
MAX_FEATURES = 10000     # Quantas palavras/termos o TF-IDF vai considerar
NGRAM_MAX    = 2         # 1 = unigramas | 2 = uni + bigramas
C            = 10.0      # Parâmetro de regularização da Regressão Logística
                         # C pequeno = mais regularização (modelo simples)
                         # C grande  = menos regularização (modelo complexo)
RUN_NAME     = "exp-02-vocab-largo"   # MUDE a cada run para identificar no DagsHub!
# ════════════════════════════════════════════════════════════════════════════

# Cria o pipeline: TF-IDF → Regressão Logística
# O Pipeline garante que o mesmo pré-processamento é aplicado
# tanto no treino quanto na inferência
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=MAX_FEATURES,
        ngram_range=(1, NGRAM_MAX),
        strip_accents="unicode",  # normaliza acentos
        sublinear_tf=True,        # aplica log ao TF para suavizar
    )),
    ("clf", LogisticRegression(
        C=C,
        max_iter=1000,
        solver="lbfgs",
    )),
])

# Treina o modelo
start_time = time.time()
pipeline.fit(X_train, y_train)
latency = time.time() - start_time
print(f"Tempo de treino: {latency:.2f} segundos")

# Avalia no conjunto de teste
start_time = time.time()
y_pred = pipeline.predict(X_test)
latency = time.time() - start_time
print(f"Tempo de inferência: {latency:.2f} segundos")

acc    = accuracy_score(y_test, y_pred)
f1     = f1_score(y_test, y_pred, average="weighted")

print("\n" + "=" * 50)
print(classification_report(y_test, y_pred))
print("=" * 50)

print(f"\nRun '{RUN_NAME}' finalizado com sucesso!")
print(f"Acurácia : {acc:.2%}")
print(f"F1 (weighted): {f1:.3f}")

Tempo de treino: 10.16 segundos
Tempo de inferência: 1.01 segundos

                precision    recall  f1-score   support

 ENTERTAINMENT       0.82      0.83      0.82      1963
      POLITICS       0.87      0.88      0.87      1962
STYLE & BEAUTY       0.89      0.85      0.87      1963
        TRAVEL       0.88      0.87      0.87      1963
      WELLNESS       0.81      0.84      0.83      1963

      accuracy                           0.85      9814
     macro avg       0.85      0.85      0.85      9814
  weighted avg       0.85      0.85      0.85      9814


Run 'exp-02-vocab-largo' finalizado com sucesso!
Acurácia : 85.32%
F1 (weighted): 0.853


### **Modelo com TinyBERT**

⚠ **Atenção!** Esse modelo precisa de um ambiente com GPU (Colab). ⛔ **Não rodar em CPU**.

#### O que é o TinyBERT?

O TinyBERT é uma versão compactada e otimizada do BERT (Bidirectional Encoder Representations from Transformers), um modelo de linguagem pré-treinado muito poderoso. Ele foi desenvolvido para ter um tamanho de modelo e tempo de inferência significativamente menores, mantendo um desempenho comparável ao do BERT original em várias tarefas de Processamento de Linguagem Natural (PLN). Isso o torna ideal para cenários onde recursos computacionais são limitados, como dispositivos móveis ou aplicações em tempo real, sem sacrificar muita precisão.

In [ ]:
# Instala as bibliotecas necessárias para rodar o modelo de linguagem
# pytorch para CPU
# ! pip install torch --index-url https://download.pytorch.org/whl/cpu
! pip install torch
! pip install transformers
! pip install datasets

In [ ]:
# Prepara os dados para o modelo de linguagem (BERT)
from datasets import Dataset, DatasetDict
from sklearn.preprocessing import LabelEncoder

# Transformar labels em números (BERT precisa de rótulos numéricos)
le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_test_num = le.transform(y_test)

# Criar os DataFrames combinados
df_train_final = pd.DataFrame({"clean_headline": X_train, "category": y_train_num})
df_test_final = pd.DataFrame({"clean_headline": X_test, "category": y_test_num})

In [ ]:
# Carregamento do Modelo (TinyBERT)
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not o

In [ ]:
# Criar o objeto Dataset do Hugging Face
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_pandas(df_train_final, preserve_index=False),
    "test": Dataset.from_pandas(df_test_final, preserve_index=False)
})

# Função de tokenização
def tokenize_fn(examples):
    return tokenizer(examples["clean_headline"], padding="max_length", truncation=True, max_length=256)

# Aplicar em tudo de uma vez
tokenized_datasets = dataset.map(tokenize_fn, batched=True)

# Limpeza para o PyTorch/GPU e renomear coluna de labels
tokenized_datasets = tokenized_datasets.remove_columns(["clean_headline"])
tokenized_datasets = tokenized_datasets.rename_column("category", "labels")
tokenized_datasets.set_format("torch")

Map:   0%|          | 0/39256 [00:00<?, ? examples/s]

Map:   0%|          | 0/9814 [00:00<?, ? examples/s]

In [48]:
from transformers import TrainingArguments, Trainer
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')

    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./results_tinybert",
    num_train_epochs=5,              # Aumentado para 5 épocas
    per_device_train_batch_size=32,  # Batch alto para aproveitar a GPU
    per_device_eval_batch_size=32,
    eval_strategy="epoch",           # Avalia o modelo a cada época
    save_strategy="epoch",           # Salva um checkpoint a cada época
    fp16=True,                       # Habilita precisão mista (muito mais rápido em GPU)
    learning_rate=2e-5,              # Taxa de aprendizado padrão para BERT
    weight_decay=0.01,
    load_best_model_at_end=True,     # Ao final, carrega a melhor versão vista no teste
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
metrics = train_result.metrics
print(f"Tempo total de treino: {metrics['train_runtime']:.2f} segundos")
print(f"Amostras processadas por segundo: {metrics['train_samples_per_second']:.2f}")

eval_metrics = trainer.evaluate()
print(f"Tempo total de teste: {eval_metrics['eval_runtime']:.2f} segundos")
print(f"F1-Score final: {eval_metrics['eval_f1']:.4f}")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.356128,0.413094,0.866110,0.865951
2,0.293668,0.408938,0.873446,0.873876
3,0.269211,0.391571,0.877012,0.877190
4,0.236546,0.405612,0.880579,0.880776
5,0.220825,0.409167,0.881598,0.881711


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias'].
There were unexpected keys in the checkp

Tempo total de treino: 674.92 segundos
Amostras processadas por segundo: 290.82


Tempo total de teste: 12.44 segundos
F1-Score final: 0.8817


#### Quantização do Modelo

A quantização dinâmica é útil para CPUs. Se você fosse usar o modelo em um hardware específico (como uma TPU ou um hardware mobile com aceleradores), outras formas de quantização (como quantização estática ou quantização ciente do treinamento) poderiam ser mais apropriadas.

In [51]:
from transformers import pipeline
from torch.quantization import quantize_dynamic

# Criar um pipeline para classificação de texto usando o modelo treinado
# (Você pode carregar o modelo salvo anteriormente se preferir)
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1 # Use GPU se disponível, senão CPU
)

# Exemplo de quantização dinâmica para CPU
# Isso irá quantizar os pesos do modelo para int8, reduzindo o tamanho e acelerando a inferência na CPU
# Note que a quantização dinâmica é aplicada geralmente após o treinamento
quantized_model = quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)

print("Modelo original:\n", model)
print("\nModelo quantizado:\n", quantized_model)

# Opcional: Salvar o modelo quantizado
# from huggingface_hub import HfFolder
# quantized_model.save_pretrained("./modelo_tinybert_quantizado")
# tokenizer.save_pretrained("./modelo_tinybert_quantizado")
# print("Modelo TinyBERT quantizado e tokenizer salvos em './modelo_tinybert_quantizado'")


Modelo original:
 BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 312, padding_idx=0)
      (position_embeddings): Embedding(512, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((3

/tmp/ipykernel_1853/1093261516.py:16: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)


In [53]:
import os
import torch

def get_model_size(model):
    """Calcula o tamanho de um modelo PyTorch em MB."""
    torch.save(model.state_dict(), "temp_model.pth")
    size_mb = os.path.getsize("temp_model.pth") / (1024 * 1024)
    os.remove("temp_model.pth") # Remove o arquivo temporário
    return size_mb

# Calcula o tamanho do modelo original
original_model_size = get_model_size(model)
print(f"Tamanho do Modelo Original: {original_model_size:.2f} MB")

# Calcula o tamanho do modelo quantizado
quantized_model_size = get_model_size(quantized_model)
print(f"Tamanho do Modelo Quantizado: {quantized_model_size:.2f} MB")

# Calcula a redução percentual
reduction_percent = ((original_model_size - quantized_model_size) / original_model_size) * 100
print(f"Redução de tamanho: {reduction_percent:.2f}%")

Tamanho do Modelo Original: 54.78 MB
Tamanho do Modelo Quantizado: 41.49 MB
Redução de tamanho: 24.26%


#### Salvando os modelos e o LabelEncoder

In [ ]:
# Salva o modelo original (refinado) do TinyBERT
# O 'trainer.save_model()' já salva o modelo, a configuração e o tokenizer na pasta especificada
trainer.save_model("./modelo_tinybert_refinado")
print("Modelo TinyBERT refinado e tokenizer salvos em './modelo_tinybert_refinado'")


In [ ]:
import torch

# Salva o modelo quantizado (apenas o estado, pois foi quantizado dinamicamente)
# Crie um diretório para o modelo quantizado se ele não existir
import os
if not os.path.exists("./modelo_tinybert_quantizado"):
    os.makedirs("./modelo_tinybert_quantizado")

torch.save(quantized_model.state_dict(), "./modelo_tinybert_quantizado/pytorch_model.bin")

# Salva o tokenizer também para o modelo quantizado, pois ele é o mesmo
tokenizer.save_pretrained("./modelo_tinybert_quantizado")

print("Modelo TinyBERT quantizado e tokenizer salvos em './modelo_tinybert_quantizado'")


In [ ]:
import joblib

# Salva o LabelEncoder para poder converter os rótulos numéricos de volta para texto
joblib.dump(le, "./modelo_tinybert_refinado/label_encoder.joblib")
joblib.dump(le, "./modelo_tinybert_quantizado/label_encoder.joblib") # Salva também para o modelo quantizado

print("LabelEncoder salvo.")


#### Como carregar os modelos e o LabelEncoder

In [ ]:
# Exemplo de como carregar o modelo refinado:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import joblib

# loaded_tokenizer = AutoTokenizer.from_pretrained("./modelo_tinybert_refinado")
# loaded_model = AutoModelForSequenceClassification.from_pretrained("./modelo_tinybert_refinado")
# loaded_le = joblib.load("./modelo_tinybert_refinado/label_encoder.joblib")

# Exemplo de como carregar o modelo quantizado:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import torch
# import joblib
# from torch.quantization import quantize_dynamic

# loaded_quantized_tokenizer = AutoTokenizer.from_pretrained("./modelo_tinybert_quantizado")
# # Para carregar o modelo quantizado, primeiro instancie o modelo original e então carregue o state_dict quantizado
# loaded_quantized_model = AutoModelForSequenceClassification.from_pretrained("huawei-noah/TinyBERT_General_4L_312D", num_labels=NUM_LABELS)
# loaded_quantized_model = quantize_dynamic(loaded_quantized_model, {torch.nn.Linear}, dtype=torch.qint8)
# loaded_quantized_model.load_state_dict(torch.load("./modelo_tinybert_quantizado/pytorch_model.bin"))
# loaded_quantized_le = joblib.load("./modelo_tinybert_quantizado/label_encoder.joblib")
